In [35]:
import pandas as pd
import re

markdown_string = """
| TokenMixer    | Kernel Size | JSRT           | Graz            | Tiger@768²                         |
|:--------------|-------------|----------------|-----------------|------------------------------------|
| pooling       | 3           | 0.9468, 7.8257 | 0.8094, 9.3323  |                                    |
|               | 5           | 0.9463, 8.2299 | 0.8072, 9.4621  |                                    |
|               | 7           | 0.945, 8.192   | 0.7973, 9.5939  |                                    |
|               | 9           | -              | -               |                                    |
| conv          | 3           | 0.9486, 8.2723 | 0.8164, 9.6074  |                                    |
|               | 5           | 0.951, 8.8047  | 0.8276, 8.6283  |                                    |
|               | 7           | 0.9504, 8.5596 | 0.833, 7.2717   |                                    |
|               | 9           | -              | -               |                                    |
| sep_conv      | 3           | 0.95, 8.1622   | 0.8119, 9.238   |                                    |
|               | 5           | 0.9495, 8.5882 | 0.8319, 8.1215  |                                    |
|               | 7           | 0.9504, 8.5015 | 0.832, 7.6763   |                                    |
|               | 9           | -              | -               |                                    |
| locAttn       | 3           | 0.9418, 7.6488 | 0.8165, 10.3977 |                                    |
|               | 5           | 0.9465, 8.094  | 0.7957, 9.055   |                                    |
|               | 7           | 0.9449, 7.1625 | 0.7927, 9.1572  |                                    |
|               | 9           | -              | -               |                                    |
| fullAttn      | -           | 0.9443, 7.5997 | 0.7562, 20.5518 | - |
| identity      | -           | 0.9458, 7.7736 | 0.7417, 19.0719 | 0.5358, 535.9729                   |
| UNet          | 3           | 0.9552, 5.1958 | 0.8478, 13.1739 | 0.5666, 565.8803                   |
|               | 5           | 0.9499, 6.9125 | 0.8258, 16.4765 | 0.5458, 551.4406                   |
|               | 7           | 0.9505, 5.5495 | 0.8318, 17.5944 | 0.5533, 553.0115                   |
|               | 9           | -              | -               | 0.5544, 551.8779                   |
| UNet@PatchEmb | 3           | 0.9357, 8.291  | 0.7738, 10.5785 | 0.5735, 589.6542                   |
|               | 5           | 0.9342, 7.1374 | 0.7864, 8.7964  | 0.5981, 540.7787                   |
|               | 7           | 0.9335, 8.3351 | 0.7629, 9.6254  | 0.6047, 499.9212                   |
|               | 9           | -              | -               | 0.5639, 549.7401                   |

"""

lines = markdown_string.split("\n")
header = lines[1].strip("|").split("|")
header = list(map(lambda s: s.strip(), header))

data = []

# Loop through lines starting from 2
for line in lines[3:]:

    # Break once we hit an empty line
    if not line.strip():
        break

    cols = line.strip("|").split("|")
    cols = map(lambda s: s.strip(), cols)
    row = dict(zip(header, cols))
    data.append(row)

df = pd.DataFrame(data)
df.iloc[:, 0] = df.iloc[:, 0].replace("", pd.NA).ffill()
df.set_index(['TokenMixer', 'Kernel Size'], inplace=True)

df_split = df.apply(lambda col: col.str.split(","))
df_expanded = pd.concat(
    [df_split[col].apply(pd.Series).add_prefix(f"{col.strip()}_") for col in df_split.columns],
    axis=1
)

suffix_map = {
    "_0": "_dsc",
    "_1": "_hdd95"
}
df = df_expanded.rename(columns=lambda col: next((col.replace(k, v) for k, v in suffix_map.items() if k in col), col))
df = df.apply(pd.to_numeric, errors="coerce")

# drop unet and other metrics
df_rank = df.loc[~df.index.get_level_values(0).str.startswith('UNet'), df.columns.str.endswith('_dsc')]
df_rank = df_rank.rank(axis=0, ascending=False, method="first")
df_rank_no_imgwoof = df_rank.iloc[:, 1:]

df_rank


JSRT_dsc  Graz_dsc  Tiger@768²_dsc
TokenMixer Kernel Size                                    
pooling    3                 7.0       8.0             NaN
           5                 9.0       9.0             NaN
           7                11.0      10.0             NaN
           9                 NaN       NaN             NaN
conv       3                 6.0       6.0             NaN
           5                 1.0       4.0             NaN
           7                 2.0       1.0             NaN
           9                 NaN       NaN             NaN
sep_conv   3                 4.0       7.0             NaN
           5                 5.0       3.0             NaN
           7                 3.0       2.0             NaN
           9                 NaN       NaN             NaN
locAttn    3                14.0       5.0             NaN
           5                 8.0      11.0             NaN
           7                12.0      12.0             NaN
           9                 NaN       NaN             NaN
fullAttn   -                13.0      13.0             NaN
identity   -                10.0      14.0             1.0

# Pool Size ranking

In [36]:
df_rank.groupby('Kernel Size').mean()

,JSRT_dsc,Graz_dsc,Tiger@768²_dsc
Kernel Size,,,
-,11.50,13.50,1.0
3,7.75,6.50,NaN
5,5.75,6.75,NaN
7,7.00,6.25,NaN
9,NaN,NaN,NaN


# Token Mixer

In [37]:
df_rank.groupby('TokenMixer').mean()

,JSRT_dsc,Graz_dsc,Tiger@768²_dsc
TokenMixer,,,
conv,3.000000,3.666667,NaN
fullAttn,13.000000,13.000000,NaN
identity,10.000000,14.000000,1.0
locAttn,11.333333,9.333333,NaN
pooling,9.000000,9.000000,NaN
sep_conv,4.000000,4.000000,NaN


In [39]:
df_rank.groupby('TokenMixer').mean().mean(1).sort_values()

TokenMixer
conv         3.333333
sep_conv     4.000000
identity     8.333333
pooling      9.000000
locAttn     10.333333
fullAttn    13.000000
dtype: float64